In [0]:
%run ./00_Config_and_Logging

In [0]:
# ==========================================
# RUN SETTINGS for the whole pipeline: edit the values, then run the next cell
#   MODE        "incremental" -> process whatever is new in every layer
#   MODE        "backfill"    -> replay START_DATE..END_DATE through every layer
#   START_DATE / END_DATE  "YYYY-MM-DD" (backfill only; END_DATE defaults to START_DATE)
# Then run the remaining cells one by one (or Run all): Bronze -> Silver -> Gold.
# ==========================================
MODE = "incremental"
START_DATE = ""
END_DATE = ""

In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import col

# The layer notebooks read this instead of their own SETTINGS cell
PIPELINE_SETTINGS = validate_run_settings(
    {"mode": MODE, "start_date": START_DATE, "end_date": END_DATE}, layer="pipeline")
PIPELINE_START = datetime.now()

def check_step(layer):
    """Stop the pipeline if the layer that just ran logged a Failure."""
    bad = (spark.read.format("delta").load(log_table_path)
           .filter((col("layer_processed") == layer) & (col("start_time") >= F.lit(PIPELINE_START))
                   & (col("status") == "Failure")))
    if bad.count() > 0:
        display(bad)
        raise Exception(f"{layer} reported failures. Stopping before the next layer.")
    print(f"{layer}: no failures.")

In [0]:
%run ./01_Raw_to_Bronze

In [0]:
check_step("Raw-to-Bronze")

In [0]:
%run ./02_Bronze_to_Silver

In [0]:
check_step("Bronze-to-Silver")

In [0]:
%run ./03_Silver_to_Gold

In [0]:
check_step("Silver-to-Gold")
print("Pipeline finished: Bronze -> Silver -> Gold.")
display(spark.read.format("delta").load(log_table_path)
        .filter(col("start_time") >= F.lit(PIPELINE_START)).orderBy(col("start_time").desc()))